# Contexto de cambios semánticos de palabras en proyecto de ley de salud


**Objetivo**

Se analizan los títulos de proyectos de ley de salud presentados entre 2009 y 2024 correspondientes a los períodos de sesiones 127 a 141 en la Cámara de Diputados de la Nación Argentina. Para más información [sesiones](https://www.hcdn.gob.ar/sesiones/index.html) .

Un proyecto es de ley sobre salud, si tiene indicado el tipo de proyecto como "LEY" y el primer giro a comisión (comisión cabecera) es a una comisión vinculada a salud ('accion social y salud publica', 'asistencia social y salud publica', 'salud' o 'salud y deporte').

**Método**

Se identificaron 2848 proyectos de ley sobre salud. Se limpiaron los textos de los títulos de los proyectos. Se generaron corpus (colección cuidadosamente seleccionada y organizada de textos) para períodos quinquenales (2009,2014 y2019). 

    Período 2009: Periodos parlamentarios 127-131 (03/2009 - 02/2014)
    Período 2014: Periodos parlamentarios 132-136 (03/2014 - 02/2019)
    Período 2019: Periodos parlamentarios 137-141 (03/2019 - 02/2024)

Se construyeron modelos de word embedding por períodos a comparar. Word embedding es un método utilizado en el procesamiento del lenguaje natural para representar palabras o documentos como vectores numéricos. La selección de parámetros para un modelo word embedding en corpus pequeños se baso en la evaluación sobre una muestra de palabras en los distintos períodos. Determinada la combinación de parámetros para los corpus pequeños, se continuo con la identificación de enfoque estable para la deteccion de cambios semánticos de palabras entre periodos quinquenales consecutivos, es decir cambios de palabras entre 2009 vs 2014 y 2014 vs 2019.
Se exploraron dos enfoques, Procrustes y NN. Para analizar la estabilidad de enfoque se evaluó cuantitativa y cualitativamente ambos enfoques y comparó. 

Aplicamos la comparación a la tarea de detección de cambios en el uso de palabras por par de período comparado y enfoque. 
Utilizamos la intersección @k, propuesta por Gonen et al., que mide el porcentaje de palabras compartidas en las k palabras más cambiadas durante varios reinicios, cambiando cada vez la semilla aleatoria.

Calculamos las palabras comunes cambiadas en las repeticiones. Específicamente, ejecutamos cada enfoque de cambio de uso y recopilamos las k palabras más cambiadas, donde k ∈ k = [50,  100,  150,  200,  250,  300,  350,  400,  450,  500,  550,
        600,  650,  700,  750,  800,  850,  900,  950, 1000]. Luego, para cada uno de las combinaciones de elementos tomados de 2 en 2  de ejecuciones diferentes y para cada uno de los valores de k, medimos el porcentaje de palabras compartidas en las predicciones de palabras con mayor cambio de cada enfoque. **Un valor de cero entre un par de ejecuciones significa que no hay palabras compartidas en sus predicciones, lo que indica alta variabilidad, mientras que un valor de uno indica alta estabilidad.**

Intersección @k: mide el porcentaje de palabras compartidas en las predicciones de k palabras con mayor cambio por par de ejecuciones.
* Valor cercano a 1.0 : alta estabilidad, resultados consistentes.
* Valor cercano a 0.0 : alta variabilidad, resultados sensibles a la semilla.

Se determino que el enfoque de Procruste para k=250 palabras es el modelo que permite analizar el cambio semántico de palabras entre pares de períodos quinquenales. Para cada palabra, se calcula el promedio de similaridad coseno (medida de estabilidad del modelo) Este valor es estimado junto con los respectivos intervalos de confianza del 95%, utilizando el método bootstrap.


**Config modelo word embedding**

https://radimrehurek.com/gensim/models/word2vec.html

* tam_vector = 50  # Dimensionalidad de los vectores de palabras.
* ventana = 5 # ventana ( int , opcional ): distancia máxima entre la palabra actual y la palabra prevista dentro de una oración.
* min_frec = 2 #  Ignora todas las palabras con una frecuencia total menor que esta.
* w = 1  # utilice estos muchos subprocesos de trabajo para entrenar el modelo (=entrenamiento más rápido con máquinas de múltiples núcleos).
* s = 1 # Algoritmo de entrenamiento: 1 para skip-gram; de lo contrario, CBOW.
* e = 100 # Número de iteraciones (épocas) en el corpus. (Anteriormente: iter )
* semilla = 1 # seed ( int , opcional ): Semilla para el generador de números aleatorios. 
* iteracion = 50


* Par de periodos (2009, 2014) : Esto indica los pares de periodos comparados [2009, 2014) vs  [2014, 2019)
* Par de periodos (2014, 2019) : Esto indica los pares de periodos comparados [2014, 2019) vs [2019, 2024)




 

In [2]:
# Importar librería
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import pickle
import os
import re
import random as rn
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import bootstrap
import matplotlib.pyplot as plt
import itertools
from scipy.spatial.distance import cosine
from scipy import stats

from ast import literal_eval
from gensim.models import Word2Vec
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
tqdm.pandas()


In [3]:
# Configurar
load_dotenv() # Cargar las variables de entorno del archivo .env
BASE_DIR =  os.getenv("DIR_BASE")
RESULTADOS_DIR = os.getenv("DIR_DATOS_PROCESADOS") # Acceder a las variables de entorno
modelo_dir =  RESULTADOS_DIR+ '/archivos_out/modelos_swmwosge_5051100'
pd.set_option('display.max_colwidth', None)

In [4]:
RESULTADOS_DIR

'C:/Users/Usuario/iamas_iniciativasleg_salud/src/data/'

In [5]:
basename = 'NN'
with open(modelo_dir+'/top250_periodos_'+basename+'_df.pkl', 'rb') as file: 
    topn_periodo = pickle.load(file) ## Diccionario

### Distribución entre períodos


Para determinar que una palabra tiene cambio semántico entre los pares de periodos comparados (período 2009 vs período 2014 y período 2014 vs período 2019) se debe garantizar que :
* Se identifica la palabra como palabra consistentemente cambiantes en alguno de los pares de períodos comparados.
* El promedio de similaridad coseno asociado a la palabra es menor a 0.5 tal que:
    * Cambio fuerte: promedio de similaridad coseno  < 0.3
    * Cambio moderado: promedio de similaridad coseno entre 0.3 y 0.5

In [6]:
# Abrir
with open(os.getenv("DIR_DATOS_PROCESADOS") + 'base_texto_df_ley_0924.pkl', 'rb') as file: 
    basetexto_df = pickle.load(file)
basetotaltexto_df = basetexto_df.copy()  # todas las iniciativas
basetexto_df = basetexto_df[basetexto_df['Proyecto_SALUD']==1]
print("Conj. de datos de proyecto_SALUD con Nan en Período:",basetexto_df.shape)
basetexto_df = basetexto_df[~basetexto_df['Periodo'].isna()]
print("Conj. de datos de proyecto_SALUD sin Nan en Período:",basetexto_df.shape)
basetexto_df['Periodo_5anios'] = 2009 # el período parlamentario  127 inicia el 01/03/2009
basetexto_df.loc[(basetexto_df['Periodo']>=132) & (basetexto_df['Periodo']<137), 'Periodo_5anios'] =  2014 # 132 inicia el 01/03/2014
basetexto_df.loc[(basetexto_df['Periodo']>=137), 'Periodo_5anios'] =  2019 # 137 inicia el 01/03/2019 

Conj. de datos de proyecto_SALUD con Nan en Período: (2848, 12)
Conj. de datos de proyecto_SALUD sin Nan en Período: (2848, 12)


In [7]:
def mostrar_ejemplos_contextuales_titulos2(df, palabra, periodo, n=5):
    """
    Versión mejorada que siempre retorna exactamente n elementos
    """
    try:
        # Validaciones iniciales
        if df.empty or pd.isna(palabra) or pd.isna(periodo):
            return ["Datos inválidos"] * n
        
        # Filtrar por periodo
        df_periodo = df[df['Periodo_5anios'] == periodo]
        
        if df_periodo.empty:
            return [f"No hay datos para periodo {periodo}"] * n
        
        # Filtrar títulos que contienen la palabra
        mask = df_periodo['Título normalizado'].str.lower().str.contains(
            palabra.lower(), na=False, regex=False
        )
        titulos_con_palabra = df_periodo[mask]
        
        if titulos_con_palabra.empty:
            return [f"No se encontró '{palabra}' en periodo {periodo}"] * n
        
        # Obtener ejemplos
        disponibles = min(n, len(titulos_con_palabra))
        ejemplos = titulos_con_palabra['Título'].str.lower().sample(
            n=disponibles, 
            random_state=42,
            replace=False
        ).tolist()
        
        # Rellenar si no hay suficientes
        while len(ejemplos) < n:
            ejemplos.append(f"Ejemplo adicional no disponible ({len(ejemplos)+1}/{n})")
        
        print(f"✓ '{palabra}' en {periodo}: {disponibles} ejemplos")
        return ejemplos
        
    except Exception as e:
        print(f"✗ Error con '{palabra}' en {periodo}: {e}")
        return [f"Error: {str(e)[:30]}..."] * n



In [21]:
for par_periodo in pares_periodo:
    print(pares_periodo)
    df = topn_periodo[par_periodo].sort_values('similaridad_semantica_mean', ascending=False).head(10)
    df['Ejemplos '+str(par_periodo[0])] = None
    df['Ejemplos '+str(par_periodo[1])] = None
    for index, row in df.iterrows():
        palabra = row['palabra']
        # Obtener los ejemplos contextuales
        ejemplos = mostrar_ejemplos_contextuales_titulos2(basetexto_df, palabra, par_periodo[0], n=5)
        df.loc[index, 'Ejemplos '+str(par_periodo[0])] = str(ejemplos)
        ejemplos = mostrar_ejemplos_contextuales_titulos2(basetexto_df, palabra, par_periodo[1], n=5)
        df.loc[index, 'Ejemplos '+str(par_periodo[1])] = str(ejemplos)
    df.to_csv(RESULTADOS_DIR+'NN_top20_'+str(par_periodo[0])+'_Ejemplo.csv', index=False)

[(2009, 2014), (2014, 2019)]
✓ 'acompanante' en 2009: 3 ejemplos
✓ 'acompanante' en 2014: 5 ejemplos
✓ 'dengue' en 2009: 5 ejemplos
✓ 'dengue' en 2014: 5 ejemplos
✓ 'legal' en 2009: 3 ejemplos
✓ 'legal' en 2014: 5 ejemplos
✓ 'precio' en 2009: 1 ejemplos
✓ 'precio' en 2014: 5 ejemplos
✓ 'evaluacion' en 2009: 5 ejemplos
✓ 'evaluacion' en 2014: 5 ejemplos
✓ 'materia' en 2009: 5 ejemplos
✓ 'materia' en 2014: 5 ejemplos
✓ 'periodo' en 2009: 5 ejemplos
✓ 'periodo' en 2014: 2 ejemplos
✓ 'ter' en 2009: 5 ejemplos
✓ 'ter' en 2014: 5 ejemplos
✓ 'premio' en 2009: 2 ejemplos
✓ 'premio' en 2014: 5 ejemplos
✓ 'discapacidad' en 2009: 5 ejemplos
✓ 'discapacidad' en 2014: 5 ejemplos
[(2009, 2014), (2014, 2019)]
✓ 'adecuado' en 2014: 2 ejemplos
✓ 'adecuado' en 2019: 5 ejemplos
✓ 'adquirido' en 2014: 4 ejemplos
✓ 'adquirido' en 2019: 5 ejemplos
✓ 'fortalecimiento' en 2014: 5 ejemplos
✓ 'fortalecimiento' en 2019: 3 ejemplos
✓ 'internacion' en 2014: 5 ejemplos
✓ 'internacion' en 2019: 5 ejemplos
✓ 'domicil

In [22]:
# Convertir datos en DataFrame para visualización
def plot_heatmap(cambios_df, title, top = 10):
    #words, values = zip(*changed_words[:top])  # Tomamos el top 10
    #df = pd.DataFrame({"Palabra": words, "Cambio Semántico": values})
    df = cambios_df.copy()
    plt.figure(figsize=(14, 10))
    sns.heatmap(df.set_index("palabra").T, cmap="Reds_r", annot=True, fmt=".2f", linewidths=0.5)
    plt.title(title)
    plt.show()

def diferencia(par):
    return par[1] - par[0]

In [15]:

palabra = 'acompanante'
df_periodo = basetexto_df[basetexto_df['Periodo_5anios'] == 2014]
# Filtrar títulos que contienen la palabra
mask = df_periodo['Título normalizado'].str.lower().str.contains(
            palabra.lower(), na=False, regex=False
)
titulos_con_palabra = df_periodo[mask]
titulos_con_palabra['Título'].str.lower()

11129                                            ejercicio profesional y regulacion de la actividad del acompañante terapeutico. regimen.
12615    ejercicio profesional y regulacion de la actividad del acompañante terapeutico. regimen (reproduccion del expediente 2305-d-16).
13585                                                                        ejercicio profesional del acompañante terapeutico. regimen. 
16194                                                                                                   acompañante terapeutico. regimen.
16522                                           ejercicio profesional y regulacion de la actividad del acompañante terapeutico. regimen. 
17627                                            ejercicio profesional y regulacion de la actividad del acompañante terapeutico. regimen.
21927                                                                         ejercicio profesional del acompañante terapeutico. regimen.
Name: Título, dtype: object

In [16]:
titulos_con_palabra

,Proyecto.ID,Título,Título procesado,Título normalizado,Cant_token,Cant_token_procesados,Cant_token_normalizado,Proyecto_SALUD,Resultado,Periodo,Año,Tokens,Periodo_5anios
11129,HCDN222282,EJERCICIO PROFESIONAL Y REGULACION DE LA ACTIVIDAD DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN.,ejercicio profesional y regulacion de la actividad del acompanante terapeutico regimen,ejercicio profesional regulacion actividad acompanante terapeutico regimen,11,11,7,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,136.0,2018,"[ejercicio, profesional, regulacion, actividad, acompanante, terapeutico, regimen]",2014
12615,HCDN213070,EJERCICIO PROFESIONAL Y REGULACION DE LA ACTIVIDAD DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN (REPRODUCCION DEL EXPEDIENTE 2305-D-16).,ejercicio profesional y regulacion de la actividad del acompanante terapeutico regimen reproduccion del expediente numero d numero,ejercicio profesional regulacion actividad acompanante terapeutico regimen,15,17,7,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,136.0,2018,"[ejercicio, profesional, regulacion, actividad, acompanante, terapeutico, regimen]",2014
13585,HCDN207470,EJERCICIO PROFESIONAL DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN.,ejercicio profesional del acompanante terapeutico regimen,ejercicio profesional acompanante terapeutico regimen,6,6,5,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,135.0,2017,"[ejercicio, profesional, acompanante, terapeutico, regimen]",2014
16194,HCDN192769,ACOMPAÑANTE TERAPEUTICO. REGIMEN.,acompanante terapeutico regimen,acompanante terapeutico regimen,3,3,3,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,134.0,2016,"[acompanante, terapeutico, regimen]",2014
16522,HCDN191148,EJERCICIO PROFESIONAL Y REGULACION DE LA ACTIVIDAD DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN.,ejercicio profesional y regulacion de la actividad del acompanante terapeutico regimen,ejercicio profesional regulacion actividad acompanante terapeutico regimen,11,11,7,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,134.0,2016,"[ejercicio, profesional, regulacion, actividad, acompanante, terapeutico, regimen]",2014
17627,HCDN185439,EJERCICIO PROFESIONAL Y REGULACION DE LA ACTIVIDAD DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN.,ejercicio profesional y regulacion de la actividad del acompanante terapeutico regimen,ejercicio profesional regulacion actividad acompanante terapeutico regimen,11,11,7,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,134.0,2016,"[ejercicio, profesional, regulacion, actividad, acompanante, terapeutico, regimen]",2014
21927,HCDN164757,EJERCICIO PROFESIONAL DEL ACOMPAÑANTE TERAPEUTICO. REGIMEN.,ejercicio profesional del acompanante terapeutico regimen,ejercicio profesional acompanante terapeutico regimen,6,6,5,1.0,NO TUVO TRATAMIENTO POSTERIOR NI DICTAMEN,132.0,2014,"[ejercicio, profesional, acompanante, terapeutico, regimen]",2014
